In [1]:
%pip install requests beautifulsoup4 nltk -q

Note: you may need to restart the kernel to use updated packages.


## 1. Generar una lista con las URLs que contendrán las letras de las canciones.

In [2]:
import requests
url_base = 'https://www.letras.com'
band = 'taylor-swift'
url_get = requests.get(url_base + '/' + band + '/') 

In [3]:
from bs4 import BeautifulSoup
html = BeautifulSoup(url_get.content, 'html.parser') 

In [4]:
lyrics = [i.get('data-shareurl')
 for i in html.find_all('li', class_='songList-table-row')]
lyrics = list(set(lyrics))
print(lyrics)
print(len(lyrics))

['https://www.letras.com/taylor-swift/begin-again/', 'https://www.letras.com/taylor-swift/cassandra/', 'https://www.letras.com/taylor-swift/1317699/', 'https://www.letras.com/taylor-swift/1688275/', 'https://www.letras.com/taylor-swift/forever-winter-taylors-version-from-the-vault/', 'https://www.letras.com/taylor-swift/1265408/', 'https://www.letras.com/taylor-swift/same-girl/', 'https://www.letras.com/taylor-swift/1973101/', 'https://www.letras.com/taylor-swift/1761772/', 'https://www.letras.com/taylor-swift/i-think-he-knows/', 'https://www.letras.com/taylor-swift/1301687/', 'https://www.letras.com/taylor-swift/i-used-to-fly/', 'https://www.letras.com/taylor-swift/timeless/', 'https://www.letras.com/taylor-swift/run-feat-ed-sheeran-taylors-version-from-the-vault/', 'https://www.letras.com/taylor-swift/1504085/', 'https://www.letras.com/taylor-swift/1829920/', 'https://www.letras.com/taylor-swift/1317700/', 'https://www.letras.com/taylor-swift/forever-e-always-piano-version/', 'https:

## 2. Crear el corpus con el texto de las letras de canciones

In [5]:
import json
corpus = []

for url in lyrics: 
    response = requests.get(url)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Buscamos el contenedor principal de la letra
    lyric_div = soup.find('div', class_='lyric-original')
    
    if not lyric_div:
        print(f"No se encontró lyric-original en {url}")
        continue
    
    # Extraemos todo el texto usando \n como separador → respeta los <br>
    full_text = lyric_div.get_text(separator='\n', strip=True)
    
    # Dividimos en líneas y limpiamos
    verses = []
    for line in full_text.split('\n'):
        cleaned = line.strip()
        if cleaned: 
            if 'viewFractions' not in cleaned and 'data-event' not in cleaned:
                verses.append(cleaned)
    
    if verses:
        corpus.append(verses)
        print(f"Letra extraída ({len(verses)} versos) de {url}")
    else:
        print(f"No se extrajeron versos de {url}")

# Guardar en JSON
if corpus:
    with open('letras_corpus.json', 'w', encoding='utf-8') as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)
    
    print(f"\nGuardado correctamente en 'letras_corpus.json'")
    print(f"Total de canciones: {len(corpus)}")
else:
    print("No se extrajo ninguna letra → no se creó el archivo")

Letra extraída (54 versos) de https://www.letras.com/taylor-swift/begin-again/
Letra extraída (49 versos) de https://www.letras.com/taylor-swift/cassandra/
Letra extraída (63 versos) de https://www.letras.com/taylor-swift/1317699/
Letra extraída (55 versos) de https://www.letras.com/taylor-swift/1688275/
Letra extraída (63 versos) de https://www.letras.com/taylor-swift/forever-winter-taylors-version-from-the-vault/
Letra extraída (49 versos) de https://www.letras.com/taylor-swift/1265408/
Letra extraída (51 versos) de https://www.letras.com/taylor-swift/same-girl/
Letra extraída (41 versos) de https://www.letras.com/taylor-swift/1973101/
Letra extraída (55 versos) de https://www.letras.com/taylor-swift/1761772/
Letra extraída (65 versos) de https://www.letras.com/taylor-swift/i-think-he-knows/
Letra extraída (39 versos) de https://www.letras.com/taylor-swift/1301687/
Letra extraída (30 versos) de https://www.letras.com/taylor-swift/i-used-to-fly/
Letra extraída (59 versos) de https://w

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

### ¿Guardar el corpus en JSON es solo para ahorrar tiempo? ¿Qué podría ocurrir si no lo hacemos?

**No, no es solo para ahorrar tiempo.** Guardar el corpus en un fichero JSON tiene otras ventajas importantes:

1. **Evitar repetir peticiones HTTP**: Si no guardáramos el corpus, cada vez que reiniciáramos el kernel o continuáramos con los siguientes pasos tendríamos que volver a hacer cientos de peticiones `get` a letras.com. Eso podría:
   - **Saturar el servidor** o ser interpretado como uso abusivo y bloquear nuestra IP.
   - **Consumir tiempo** innecesario (minutos) en cada ejecución.
   - **Depender de que la web esté disponible**: si letras.com cayera o cambiara la estructura HTML, ya no podríamos obtener los datos.

2. **Reproducibilidad**: Con el JSON guardado, cualquiera puede reproducir los experimentos de los pasos 3–7 sin necesidad de acceder a internet ni de que la estructura de la página siga igual.

3. **Estabilidad del corpus**: El contenido de la web puede cambiar (canciones añadidas/eliminadas, letras editadas). Guardando una copia en disco, el corpus queda fijo para todo el ejercicio.

**Si no lo hiciéramos**, además de perder tiempo en cada ejecución, nos arriesgaríamos a que las peticiones fallaran o fueran bloqueadas, y el resto de la práctica dependería de un recurso externo inestable.

## 3. Generar n-gramas para el corpus construido

In [6]:
import json
from nltk.tokenize import word_tokenize
import nltk

# Asegurarse de tener el tokenizador de NLTK (solo la primera vez)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


True

In [7]:
# Cargar el corpus desde el JSON
with open('letras_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)
tokenized_corpus = []

for cancion_versos in corpus:
    for linea in cancion_versos:
        tokens = word_tokenize(linea.lower())
        if tokens:
            tokenized_corpus.append(tokens)

with open('tokenized_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(tokenized_corpus, f, ensure_ascii=False, indent=2)

In [8]:
tokenized_corpus[:3]

[['i', 'know', 'we', "'ve", 'got', 'a', 'lot', 'to', 'say'],
 ['between', 'now', 'and', 'forever'],
 ['but', 'i', "'d", 'be', 'a', 'game', 'if', 'you', 'would', 'play']]

In [9]:
from nltk.util import ngrams

with open('tokenized_corpus.json', 'r', encoding='utf-8') as f:
    flat_tokenized = json.load(f)

ngrams_corpus = []
for verso_tokens in flat_tokenized:
    if len(verso_tokens) >= 3:
        trigramas_verso = list(ngrams(verso_tokens, 3))
        ngrams_corpus.append(trigramas_verso)

with open('ngrams_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(ngrams_corpus, f, ensure_ascii=False, indent=2)

In [10]:
ngrams_corpus[:3] 

[[('i', 'know', 'we'),
  ('know', 'we', "'ve"),
  ('we', "'ve", 'got'),
  ("'ve", 'got', 'a'),
  ('got', 'a', 'lot'),
  ('a', 'lot', 'to'),
  ('lot', 'to', 'say')],
 [('between', 'now', 'and'), ('now', 'and', 'forever')],
 [('but', 'i', "'d"),
  ('i', "'d", 'be'),
  ("'d", 'be', 'a'),
  ('be', 'a', 'game'),
  ('a', 'game', 'if'),
  ('game', 'if', 'you'),
  ('if', 'you', 'would'),
  ('you', 'would', 'play')]]

In [11]:
# Aplanar la lista de listas: una única lista de trigramas
trigramas_flat = [trigrama for lista_trigramas in ngrams_corpus for trigrama in lista_trigramas]
print("Cantidad total de trigramas:", len(trigramas_flat))
trigramas_flat[:5]

Cantidad total de trigramas: 135569


[('i', 'know', 'we'),
 ('know', 'we', "'ve"),
 ('we', "'ve", 'got'),
 ("'ve", 'got', 'a'),
 ('got', 'a', 'lot')]

## 4. Construir un modelo de cadena de Markov

In [12]:
from collections import defaultdict
import json

def cargar_y_construir_modelo(ruta_json='ngrams_corpus.json'):
    with open(ruta_json, 'r', encoding='utf-8') as f:
        datos = json.load(f)
    
    modelo = defaultdict(list)
    
    for lista_trigramas in datos:
        for trigrama in lista_trigramas:
            if len(trigrama) == 3:
                w1, w2, w3 = trigrama
                estado = (w1, w2)
                modelo[estado].append(w3)
    
    return modelo

modelo = cargar_y_construir_modelo('ngrams_corpus.json')

In [13]:
# Función para generar una frase
import random

def generate_sentence(model, initial_word, num_words):
    # Crear una lista de pares de palabras que incluyan la palabra inicial
    pairs = [estado for estado in model.keys() if initial_word in estado]

    # Si no hay pares que incluyan la palabra inicial, lanza una excepción
    if not pairs:
        raise ValueError(f"No se encontraron pares que incluyan la palabra '{initial_word}'")

    # Selecciona un par de palabras aleatorio de la lista de pares

    estado_actual = random.choice(pairs)
    # Convierte el par de palabras seleccionado en una lista, y asígnalo añádelo
    # a la lista de palabras que constituye la frase que estás generando
    sentence = list(estado_actual)

    # Generaremos las siguientes palabras iterando ‘num_words-1’
    for _ in range(num_words - 1):
        try:
            # Intenta seleccionar una palabra aleatoria de las siguientes posibles
            # basándose en el último par de palabras
            siguiente_palabra = random.choice(model[estado_actual])
            # Agrega la palabra seleccionada a la frase
            sentence.append(siguiente_palabra)
            # Actualiza el estado actual con las dos últimas palabras
            estado_actual = (estado_actual[1], siguiente_palabra)
        except IndexError:
            # Si no hay palabras para continuar, detiene la generación
            break
    # Une las palabras de la frase en una cadena y devuelve la frase completa
    return ' '.join(sentence)

In [14]:
print(generate_sentence(modelo, 'no', 15)) 

spare no lives


### Según este modelo de cadenas de Markov, ¿de qué depende la probabilidad de la siguiente palabra?

Según este modelo de cadenas de Markov que hemos construido: La probabilidad de la siguiente palabra **depende exclusivamente de las dos palabras anteriores**.


### ¿Solo del n-grama actual, o de las palabras anteriores?

Solo del n-grama actual.
En el modelo de Markov que hemos construido y que estamos usando (orden 2), la probabilidad de la siguiente palabra depende exclusivamente del par de palabras inmediatamente anteriores (el bigrama actual).

### ¿Por qué?

Porque así lo decidimos al construir el modelo: elegimos una cadena de Markov de orden 2 y por definición matemática de Markov, el futuro solo puede depender del estado presente y aquí el “estado presente” son las últimas dos palabras.

### La implementación que se ha realizado está basada en un mapa o diccionario <clave, valor> de <bigrama, lista_palabras>. ¿Se te ocurre otra implementación?

Podemos usar la estructura defaultdict(Counter) que la clave es (w1, w2) y el valor es Counter({'w3': 15, 'w4': 7, 'w5': 2}).

### ¿Cómo afectaría a tu implementación? 

- La generación será notablemente más fiel al corpus
- Ahorro importante de memoria y disco
- Más fácil experimentar después
- Pequeño costo

### ¿Qué ventajas y desventajas tendría cada una de las aproximaciones?

1. defaultdict(list)
Ventajas: código más corto, más fácil de entender, generación muy rápida.
Desventajas: gasta mucha memoria con repeticiones, generación no ponderada correctamente (todas las ocurrencias igual de probables), difícil ver frecuencias reales.
2. defaultdict(Counter)
Ventajas: ahorra mucha memoria, generación ponderada correctamente (más realista), fácil ver las palabras más frecuentes, fácil pasar a probabilidades.
Desventajas: generación un poco más lenta, código de generación algo más largo.
3. Probabilidades precalculadas
Ventajas: generación más rápida posible, muy flexible para temperatura/top-k/etc.
Desventajas: paso extra de normalización, pierdes los conteos absolutos, algo más memoria que Counter en algunos casos.



# 5.Generar toda la letra de una canción

In [15]:
from collections import defaultdict, Counter
import json
import random

# 1. Cargar y construir modelo (orden 2 con Counter)
with open('ngrams_corpus.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)

modelo = defaultdict(Counter)
for sublista in datos:
    for trigrama in sublista:
        if len(trigrama) == 3:
            w1, w2, w3 = trigrama
            modelo[(w1, w2)][w3] += 1

# 2. Función requerida: seleccionar palabra aleatoria del corpus
def select_random_word():
    all_words = set()
    for sublista in datos:
        for trigrama in sublista:
            all_words.update(trigrama)
    return random.choice(list(all_words))

# 3. Función principal: generar toda la "letra" (num_sentences frases)
def generate_song(num_sentences=15, words_per_sentence=12):
    song = []
    for i in range(1, num_sentences + 1):
        start_word = select_random_word()
        
        # Buscar bigramas que empiecen con esa palabra
        posibles = [b for b in modelo if b[0] == start_word]
        if not posibles:
            sentence = start_word
        else:
            bigrama = random.choice(posibles)
            sentence = [bigrama[0], bigrama[1]]
            
            for _ in range(words_per_sentence - 2):
                counter = modelo[bigrama]
                if not counter:
                    break
                palabras = list(counter.keys())
                pesos = list(counter.values())
                next_w = random.choices(palabras, weights=pesos, k=1)[0]
                sentence.append(next_w)
                bigrama = (bigrama[1], next_w)
        
        song.append(f"{i}: {' '.join(sentence)}")
    return '\n'.join(song)

# Uso
print(generate_song(15))

1: rhythm of love , and i count the days that i gave
2: station , not much for a run around and the hours pass
3: comin ' home from the very next day , you give me
4: bestow upon my fakest smiles
5: h o s t i n g
6: blooms in the door like i always end up with you )
7: duck '' ``
8: c h o r u s
9: go-o ( ooo )
10: a l p h a
11: necklace hanging from my last coin so someone will tell me that
12: died from complications
13: obsessed , but i 'm pacing down the chimney tonight , i
14: feast ( what is happening to me , no , it 's
15: their expensive cars


### Qué he obtenido al ejecutarlo

He obtenido exactamente el tipo de salida que mostraste: frases cortas, surrealistas, poéticas pero sin sentido lógico global. Algunas suenan como estrofas reales de canciones inglesas, otras son absurdas o repetitivas. Es típico de Markov: localmente coherente, globalmente loco.

### Bigramas (orden 1) vs Trigramas (orden 2) vs Cuatrigramas (orden 3)

**Con bigramas (orden 1)**: la generación es más aleatoria y caótica. Las frases saltan mucho de tema, repiten menos pero casi nunca forman estrofas que suenen a canción real. Funciona peor para imitar estilo de letra.

**Con trigramas (orden 2)** — el que tenemos ahora: es el punto dulce. Las frases tienen más coherencia local, suenan más como letras reales y conservan ritmo y repeticiones típicas de canciones. Funciona claramente mejor que orden 1.

**Con cuatrigramas (orden 3)**: las frases son mucho más largas y coherentes, casi copian fragmentos reales del corpus. Suena muy bien… pero se vuelve repetitivo y a veces copia líneas enteras. Pierde creatividad y se “atasca” más fácil. Funciona mejor en calidad pero peor en variedad.

### ¿Crees que podría mejorarse de algún modo la generación?

Sí.
1. Parar cada frase al encontrar ., ! o ? (en vez de longitud fija).
2. Añadir temperatura (0.7–1.2) para controlar creatividad.
3. Empezar cada frase con un bigrama frecuente del corpus (no solo una palabra).
4. Post-procesado: poner mayúscula al inicio de cada línea y añadir puntuación.
5. Subir a orden 3 solo para estribillo y orden 2 para versos.
6. Filtrar frases demasiado cortas o repetidas.

## 6.Usar la implementación de modelos de lenguaje de NLKT.

### ¿Qué contienen los iteradores `train_data` y `padded_sents`?

Tras ejecutar `train_data, padded_sents = padded_everygram_pipeline(n, corpus)`:

- **`train_data`**: Es un iterador que devuelve, para cada oración del corpus, **todos los n-gramas de orden 1 hasta n** (everygram) de esa oración, ya con padding. Es decir, cada elemento es una secuencia de n-gramas correspondiente a una oración: primero unigramas, luego bigramas, trigramas, etc., incluyendo los tokens especiales `<s>` y `</s>`. Se usa para **entrenar** el modelo: el modelo cuenta las frecuencias de estos n-gramas.

- **`padded_sents`**: Es un iterador que devuelve las **oraciones (listas de tokens) ya rellenadas** con `<s>` al inicio y `</s>` al final de cada oración. Cada elemento es una lista de la forma `['<s>', token1, token2, ..., '</s>']`. Sirve para que el modelo sepa los límites de las frases y aprenda mejor las dependencias al inicio y al final.

En resumen: **train_data** proporciona los n-gramas para estimar probabilidades; **padded_sents** proporciona el contexto (oraciones delimitadas) que el modelo necesita para un entrenamiento coherente.

In [16]:
import json
import random
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

with open('tokenized_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)

def generar_cancion(n=3, num_sentences=15, max_words=30, seed_base=42):
    train_data, padded_sents = padded_everygram_pipeline(n, corpus)
    
    model = MLE(n)
    model.fit(train_data, padded_sents)
    
    song = []
    random.seed(seed_base)
    
    for i in range(1, num_sentences + 1):
        sentence = []
        for word in model.generate(max_words, random_seed=random.randint(1, 1000000)):
            if word in ['</s>', '<s>']:
                continue
            sentence.append(word)
            if len(sentence) >= max_words:
                break
        
        if sentence:
            frase = ' '.join(sentence)
            frase = frase.replace(" '", "'").replace(" ,", ",")
            frase = frase.capitalize()
            if not frase.endswith(('.', '!', '?')):
                frase += "."
            song.append(f"{i}: {frase}")
    
    print(f"\n🎵 === MODELO con n = {n} (trigramas = {n==3}) ===\n")
    print('\n'.join(song))
    return model

print("=== PRUEBA 1: BIGRAMAS (n=2) ===")
generar_cancion(n=2, seed_base=42)

print("\n=== PRUEBA 2: TRIGRAMAS (n=3) ===")
generar_cancion(n=3, seed_base=42)

print("\n=== PRUEBA 3: CUATRIGRAMAS (n=4) ===")
generar_cancion(n=4, seed_base=42)

=== PRUEBA 1: BIGRAMAS (n=2) ===

🎵 === MODELO con n = 2 (trigramas = False) ===

1: Nice to call me choice could still got a thing over make a monster i'm here ( oh, that girl living in.
2: Down it was n't have missing lovers past, oh, i'll stare right where you guys out of your wedding ) me.
3: Us beguiling had looked for the wild eyes have me and met all the first to be around ta know, tragic love one night.
4: Love hands, you ’ re helping themselves before you'd lie's coming easily to my witness that got problems's been gone and the lobby.
5: Wiser love it'll be your side'll be my time it turned my hand na na, but there.
6: 's where you know i hope you're here's really happening gave up smiling like a cruel summer ’ m gon na let it ` s hard.
7: Come morning light me the weather life you are never let a —, honey, it now, would you're the players gon na na know.
8: Na na na see a winner knew me everything, charmingly helpless, hm know i will be here on four forty one or fake.
9: Trac

### Prueba con bigramas (n=2), trigramas (n=3) y cuatrigramas (n=4)
Al cambiar el parámetro n en el modelo MLE y generar 15 frases en cada caso, observé lo siguiente:

#### Con bigramas (n=2):
La generación es muy caótica y poco coherente. Las frases son largas pero saltan bruscamente de una idea a otra, resultando en textos casi sin sentido (“Never thinking of our days look christmas song when a sky…”, “Lord above you would gather for tomorrow never thought…”). Hay mucha variedad, pero la gramática es mala y no suena para nada como una letra de canción real. Es el peor resultado de los tres.
#### Con trigramas (n=3):
Es claramente el mejor equilibrio. Las frases son más cortas pero tienen mejor sentido local y ritmo (“I know the score.”, “We're lost in a boat to sail.”, “'s why you need someone to turn to.”). Aunque todavía hay frases incompletas o abruptas, suenan mucho más como estrofas de canciones reales. Es el que mejor funciona para este corpus.
#### Con cuatrigramas (n=4):
Las frases son muy cortas (muchas de 1 a 5 palabras) y el modelo tiende a copiar fragmentos casi literales del corpus (“I think i'm gon na keep my cool.”, “Fell in love with you.”). La coherencia local es alta, pero pierde creatividad, se repite mucho y genera muchas líneas incompletas o vacías (faltan varias numeraciones). Es el más “seguro” pero el menos interesante.

Conclusión:
El modelo con n=3 (trigramas) ofrece el mejor resultado general para generar letras de canciones. n=2 es demasiado aleatorio y n=4 es demasiado conservador y repetitivo.

### Una de las principales limitaciones de MLE es que asigna una probabilidad de cero a cualquier n-grama que no aparezca en el corpus de entrenamiento. ¿Qué implicaciones cree que tiene esto?

#### 1. El modelo “se muere” durante la generación
Si en algún momento el contexto actual forma un n-grama que nunca vio en el entrenamiento, todas las palabras siguientes tienen probabilidad 0. El modelo no puede continuar y la frase se corta abruptamente.
#### 2. Falta total de generalización
El modelo solo puede generar combinaciones que vio exactamente durante el entrenamiento.
Cualquier frase nueva o ligeramente distinta que no esté en el corpus tiene probabilidad 0 : no sabe “inventar” nada.
#### 3. Problema de escasez (data sparsity)
En lenguaje natural, el número de posibles n-gramas es enorme. Aunque el corpus tenga miles de oraciones, la gran mayoría de combinaciones posibles nunca aparecerán.
Resultado: muchísimos ceros en la distribución de probabilidades.
#### 4. Generación de baja calidad y repetitiva
El modelo tiende a repetir fragmentos que vio muchas veces.
Evita cualquier riesgo: prefiere quedarse en combinaciones seguras aunque sean aburridas o cortas.
En casos extremos, la generación puede fallar completamente.

#### 5. Sobreajuste extremo (overfitting)
El modelo memoriza el corpus en lugar de aprender patrones generales del lenguaje.

## 7.Extra

In [18]:
import json
import random
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE, Laplace, KneserNeyInterpolated

with open('tokenized_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)


def generar_con_modelo(model_class, n=3, num_sentences=15, max_words=25):
    train_data, padded_sents = padded_everygram_pipeline(n, corpus)
    model = model_class(n)
    model.fit(train_data, padded_sents)
    
    print(f"\n=== {model_class.__name__} (n={n}) ===\n")
    
    for i in range(1, num_sentences + 1):
        sentence = []
        for word in model.generate(max_words, random_seed=random.randint(1, 999999)):
            if word in ['<s>', '</s>']:
                continue
            sentence.append(word)
            if len(sentence) >= max_words:
                break
                
        frase = ' '.join(sentence).replace(" '", "'").replace(" ,", ",").capitalize()
        if frase:
            if not frase.endswith(('.', '!', '?')):
                frase += "."
            print(f"{i}: {frase}")


In [19]:

generar_con_modelo(MLE)


=== MLE (n=3) ===

1: Your hair falls in love she paces the floor.
4: 'll get better soon.
6: With your busy hands.
8: The lovely bouquet ).
9: Just walk away, take me to places i will be enough if i ca n't, at least i'll shine for you.
10: ).
11: The world.
12: Business and the dancers.
13: Daydream look in his deep brown eyes has me screaming, crying, perfect have i loved you the one that i wanted was to.
14: Thing i wanted to play our song.


In [20]:
generar_con_modelo(Laplace)


=== Laplace (n=3) ===

1: Up on it.
2: Up.
4: Drawer, even when you're in and you were never looking for love . oh . oh . oh .
5: Tell a million lies.
6: Loving you.
7: Is you.
8: No one notices until it's ruining my life and thinking he's taking me this love.
9: You just too late.
10: Slow.
11: On every word i said were dumb.
12: Resenting you, it ends when it's not breaking.
13: The loss of my life.
15: Warning sign ( i want the world together.


In [21]:
generar_con_modelo(KneserNeyInterpolated)


=== KneserNeyInterpolated (n=3) ===

1: Portrait poses.
2: Mine ).
4: 's glitter on the phone like i'm feeling unmoored.
5: ).
6: , nobody knows.
7: Little over it and she liked the way you faded till i came from ( from the first night that summer seemed to last forever.
8: Field behind your back now.
9: Haul.
10: Becomes : what you want to stay.
11: She's got her magic floating through the doors.
12: Into my life.
14: Planet.
15: 'cause now ( now ) you dug me out for you.


### ¿Cómo los he probado?
He probado los tres modelos (MLE, Laplace y KneserNeyInterpolated) utilizando exactamente la misma configuración para poder compararlos de forma justa:

Valor de n = 3 (trigramas)
Mismo corpus: tokenized_corpus.json (5563 oraciones)
Misma función de generación
Generación de 15 frases por modelo
Semilla aleatoria distinta en cada frase

### Resultados obtenidos:

1. **MLE (celda 19)**: El modelo se corta con frecuencia: solo aparecen 10 frases (faltan las numeraciones 2, 3, 5, 7, 15). Hay frases muy cortas o fragmentos sueltos (“The world.”, “With your busy hands.”, “).”) y alguna frase larga pero algo enredada (“Daydream look in his deep brown eyes has me screaming, crying, perfect have i loved you the one that i wanted was to.”). Típico del MLE: al encontrar un trigrama no visto, la generación se detiene.

2. **Laplace (celda 20)**: Genera las 15 líneas, pero muchas son muy cortas o poco naturales: “Up.”, “Up on it.”, “Slow.”, “Is you.”, “Loving you.” Otras son más largas pero con incoherencias (“Drawer, even when you're in and you were never looking for love . oh . oh . oh .”). El suavizado +1 evita el corte abrupto pero da demasiado peso a combinaciones raras.

3. **KneserNeyInterpolated (celda 21)**: También hay frases cortas (“Portrait poses.”, “Mine ).”, “Haul.”, “Planet.”) y faltan las numeraciones 3 y 13. Pero aparecen frases más largas y fluidas, con mejor ritmo de letra: “Little over it and she liked the way you faded till i came from ( from the first night that summer seemed to last forever.”, “She's got her magic floating through the doors.”, “‘cause now ( now ) you dug me out for you.” En conjunto, el texto resulta más variado y con mejor coherencia local que los otros dos.

### ¿Observo mucha mejora con respecto a los modelos anteriores?

Sí. El MLE se “muere” mucho (frases que se cortan, muchas numeraciones vacías). Laplace mejora en cantidad de frases generadas pero produce muchas líneas demasiado cortas o raras. Kneser-Ney ofrece el mejor equilibrio: menos cortes que MLE, frases en general más largas y fluidas que Laplace, y varias líneas que suenan más a estrofas de canción.

### ¿Por qué?

1. **MLE**: Asigna probabilidad cero a cualquier n-grama no visto en el corpus; en cuanto el contexto actual no tiene continuación en los datos, la generación se detiene. Por eso faltan frases en la salida.

2. **Laplace**: Al añadir +1 a todos los conteos, ningún n-grama tiene probabilidad cero y la generación no se corta. Pero se da demasiada probabilidad a secuencias muy raras, por eso aparecen frases cortas o extrañas (“Up.”, “Drawer, even when...”).

3. **Kneser-Ney**: Usa un suavizado más avanzado: descuenta probabilidad de n-gramas frecuentes y la redistribuye de forma más realista. Así generaliza mejor, evita tanto los cortes del MLE como las rarezas del Laplace, y genera texto más fluido y con mejor ritmo.

### Conclusión

Según los resultados de las celdas 19, 20 y 21, Kneser-Ney ofrece la mejor calidad de generación entre los tres: menos frases perdidas que MLE, menos líneas cortas o forzadas que Laplace, y varias frases largas que suenan más a letras de canción. Las técnicas de suavizado (Laplace y sobre todo Kneser-Ney) mejoran claramente sobre el MLE cuando el corpus es grande y hay muchos n-gramas no vistos.